In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
if not os.path.exists('/content/research-copilot-gnn'):
    !git clone https://github.com/shriyaasija/research-copilot-gnn.git /content/research-copilot-gnn

Cloning into '/content/research-copilot-gnn'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 35 (delta 3), reused 30 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 22.77 MiB | 37.55 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [4]:
os.chdir('/content/research-copilot-gnn')

In [5]:

!pip install -r requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.7 MB/s eta 0:00:00


In [6]:
import os
import json
import re
from pathlib import Path
import fitz  # PyMuPDF
from tqdm import tqdm

In [7]:
RAW_PDF_DIR = Path("data/raw_pdfs")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

In [8]:
import re

def clean_text(text):
    """
    Strip out non-text garbage that confuses the sentence transformer.
    PDFs sometimes contain binary blobs, weird unicode, or null bytes.
    """
    if not text or not isinstance(text, str):
        return "unknown paper"

    # Remove null bytes and non-printable characters
    text = text.replace('\x00', ' ')
    text = re.sub(r'[^\x20-\x7E\n]', ' ', text)  # keep only printable ASCII

    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # If text is too short after cleaning, use a placeholder
    if len(text) < 10:
        return "graph neural network research paper"

    # Truncate to 512 chars (sentence transformers have a token limit anyway)
    return text[:512]

In [9]:
def extract_text_from_pdf(pdf_path):
    """
    Extract text and basic metadata from a PDF file.
    Returns a dictionary with title, authors, abstract, body, references.
    """
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num, page in enumerate(doc):
        text = page.get_text()
        full_text += text + "\n"

    doc.close()

    # Split into sections (simple heuristic)
    lines = full_text.split('\n')
    lines = [l.strip() for l in lines if l.strip()]

    # Try to extract abstract (look for the word "Abstract")
    abstract = ""
    abstract_start = -1
    for i, line in enumerate(lines):
        if line.lower().startswith("abstract"):
            abstract_start = i
            break

    if abstract_start >= 0:
        # Take the next ~10 lines as abstract
        abstract_lines = []
        for line in lines[abstract_start+1:abstract_start+15]:
            if line.lower() in ["introduction", "1 introduction", "1. introduction"]:
                break
            abstract_lines.append(line)
        abstract = " ".join(abstract_lines)

    # Extract references section
    references_text = ""
    ref_start = -1
    for i, line in enumerate(lines):
        if line.lower() in ["references", "bibliography", "works cited"]:
            ref_start = i
            break

    if ref_start >= 0:
        references_text = " ".join(lines[ref_start+1:])

    return {
        "filename": pdf_path.name,
        "title": pdf_path.stem,  # Use filename as title for now
        "full_text": full_text[:5000],  # First 5000 chars
        "abstract": abstract,
        "references_raw": references_text[:3000]
    }


In [10]:
papers = []

pdf_files = list(RAW_PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs")

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    try:
        paper_data = extract_text_from_pdf(pdf_path)
        papers.append(paper_data)
        print(f"  Extracted: {pdf_path.name} ({len(paper_data['full_text'])} chars)")
    except Exception as e:
        print(f"  ERROR on {pdf_path.name}: {e}")

print(f"\nSuccessfully processed {len(papers)} papers")

Found 17 PDFs


Extracting PDFs:   6%|▌         | 1/17 [00:00<00:01,  8.10it/s]

  Extracted: paper5.pdf (5000 chars)


Extracting PDFs:  12%|█▏        | 2/17 [00:00<00:01,  9.00it/s]

  Extracted: paper12.pdf (5000 chars)


...

  Extracted: paper1.pdf (5000 chars)
  Extracted: paper16.pdf (5000 chars)


Extracting PDFs:  88%|████████▊ | 15/17 [00:02<00:00,  6.05it/s]

  Extracted: paper7.pdf (5000 chars)
  Extracted: paper4.pdf (5000 chars)


Extracting PDFs: 100%|██████████| 17/17 [00:02<00:00,  6.24it/s]

  Extracted: paper9.pdf (5000 chars)
  Extracted: paper6.pdf (5000 chars)

Successfully processed 17 papers


In [11]:
output_path = PROCESSED_DIR / "papers_extracted.json"
with open(output_path, "w") as f:
    json.dump(papers, f, indent=2)

print(f"Saved to {output_path}")

Saved to data/processed/papers_extracted.json


In [26]:
!pip install model2vec keybert --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 2.5 MB/s eta 0:00:00


In [27]:
from keybert import KeyBERT
from model2vec import StaticModel
import re

In [28]:
embedding_model = StaticModel.from_pretrained("minishlab/potion-base-8M")
kw_model = KeyBERT(embedding_model)
print("KeyBERT loaded with Model2Vec backend ✅")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

KeyBERT loaded with Model2Vec backend ✅


In [29]:
def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = text.replace('\x00', ' ')
    text = re.sub(r'[^\x20-\x7E\n]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:512]

def extract_keyphrases(text, top_n=10):
    text = clean_text(text)
    if not text or len(text) < 50:
        return []
    try:
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 3),
            stop_words='english',
            use_mmr=True,
            diversity=0.5,
            top_n=top_n
        )
        return [kw[0] for kw in keywords]
    except Exception as e:
        print(f"  Error: {e}")
        return []

In [30]:
print("Extracting keyphrases...")
for paper in tqdm(papers, desc="Keyphrases"):
    text_to_use = paper["abstract"] if paper["abstract"] else paper["full_text"]
    paper["keyphrases"] = extract_keyphrases(text_to_use, top_n=10)

# Verify it worked
print("\nSample keyphrases:")
for paper in papers[:3]:
    print(f"\n{paper['filename']}:")
    print(f"  {paper['keyphrases']}")

# Save
with open(PROCESSED_DIR / "papers_with_keyphrases.json", "w") as f:
    json.dump(papers, f, indent=2)
print("\nSaved!")

Extracting keyphrases...



Keyphrases: 100%|██████████| 17/17 [00:00<00:00, 31.48it/s]


Sample keyphrases:

paper5.pdf:
  ['automated neural architecture', 'crucial aspect progress', 'learning enabled remarkable', 'architectures currently employed', 'recognition speech', 'years variety tasks', 'existing work eld', 'time consuming error', 'machine translation', 'search methods']

paper12.pdf:
  ['convolutional neural networks', 'embedding represented graphs', 'cnns low dimensional', 'dimensional regular grids', 'ef cient numerical', 'context spectral', 'nec essary mathematical', 'design fast localized', 'work interested generalizing', 'domains social']

paper8.pdf:
  ['recurrence convolutions entirely', 'models connect encoder', 'sequence transduction models', 'decoder best performing', 'neural networks', 'transformer based solely', 'attention mechanisms', 'experiments machine', 'superior quality paralleliz', 'dispensing']

Saved!
